# 03 — SageMaker Feature Store (Feature Group + Ingestion)

This notebook:
- Reads curated CSV from S3 (produced in Notebook 01)
- Creates supervised lag features
- Creates a **SageMaker Feature Store Feature Group**
- Ingests the features (offline + optional online store)

This satisfies the course demo requirement: **feature store and feature groups**.


In [2]:
%pip install -q -r ../docker/requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

import time
import boto3
import sagemaker
import pandas as pd
import s3fs

from src.feature_engineering import make_supervised
from src.feature_engineering import time_split  # optional
from sagemaker.feature_store.feature_group import FeatureGroup

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [4]:
# Load from previous notebook
%store -r bucket
%store -r region
%store -r S3_PREFIX_PARQUET
%store -r S3_PREFIX_CSV
%store -r BUOY_IDS
%store -r manifest_s3_uri

print("Bucket:", bucket)
print("Region:", region)
print("S3_PREFIX_PARQUET:", S3_PREFIX_PARQUET)
print("S3_PREFIX_CSV:", S3_PREFIX_CSV)
print("BUOY_IDS:", BUOY_IDS)
print("Manifest (S3):", manifest_s3_uri)

Bucket: sagemaker-us-east-1-318401170150
Region: us-east-1
S3_PREFIX_PARQUET: curated/ndbc_parquet
S3_PREFIX_CSV: curated/ndbc_csv
BUOY_IDS: ['46086', '46042', '46011']
Manifest (S3): s3://sagemaker-us-east-1-318401170150/manifests/buoy=all/curated_manifest.csv


In [5]:
# Read all curated CSV from S3
fs = s3fs.S3FileSystem()

csv_glob = f"s3://{bucket}/{S3_PREFIX_CSV}/buoy=*/stdmet.csv"
csv_paths = fs.glob(csv_glob.replace("s3://",""))  # s3fs glob expects bucket/key without scheme

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found at {csv_glob}")

dfs = []
for key in sorted(csv_paths):
    s3_uri = "s3://" + key
    df = pd.read_csv(s3_uri)
    dfs.append(df)

all_data = pd.concat(dfs, ignore_index=True)
all_data["timestamp"] = pd.to_datetime(all_data["timestamp"], errors="coerce", utc=True)
all_data = all_data.dropna(subset=["timestamp"]).sort_values(["station_id","timestamp"])

print("Rows:", len(all_data))
print("Buoys:", all_data["station_id"].nunique())
all_data.head()

Rows: 117632
Buoys: 3


,timestamp,station_id,wind_direction,wind_speed,wind_gust,wave_height,dominant_wave_period,average_wave_period,mean_wave_direction
0,2023-01-01 00:10:00+00:00,46011,167.0,10.0,12.2,2.30,11.43,6.14,275.0
1,2023-01-01 00:40:00+00:00,46011,168.0,11.1,13.4,2.92,14.81,7.15,284.0
2,2023-01-01 01:10:00+00:00,46011,170.0,11.4,13.7,2.60,11.43,6.35,287.0
3,2023-01-01 01:40:00+00:00,46011,174.0,11.1,13.6,2.33,14.81,5.81,262.0
4,2023-01-01 02:10:00+00:00,46011,193.0,10.6,13.1,2.66,14.81,6.43,276.0


In [6]:
# Build supervised features (lags + target)
df_features, feature_cols = make_supervised(
    all_data,
    entity_col="station_id",
    time_col="timestamp",
    lead_hours=1,
    lags=(1, 2, 3, 6),
    include_current_features=False,
)

# Feature Store metadata
df_features["event_time"] = (
    pd.to_datetime(df_features["timestamp"], utc=True).astype("int64") / 1e9
).astype(float)

df_features["record_id"] = (
    df_features["station_id"].astype(str)
    + "_"
    + df_features["event_time"].astype(int).astype(str)
)

# Keep timestamp for debug, but you can drop it if you want
print("Supervised rows:", len(df_features))
df_features.head()

Supervised rows: 117611


,timestamp,station_id,wind_direction,wind_speed,wind_gust,wave_height,dominant_wave_period,average_wave_period,mean_wave_direction,E_star,...,wave_height_lag_3,E_star_lag_3,wind_speed_lag_3,dominant_wave_period_lag_3,wave_height_lag_6,E_star_lag_6,wind_speed_lag_6,dominant_wave_period_lag_6,event_time,record_id
0,2023-01-01 03:10:00+00:00,46011,298.0,8.3,11.6,2.71,11.43,6.84,272.0,7.3441,...,2.33,5.4289,11.1,14.81,2.30,5.2900,10.0,11.43,1.672543e+09,46011_1672542600
1,2023-01-01 03:40:00+00:00,46011,298.0,9.4,12.0,2.79,13.79,6.92,282.0,7.7841,...,2.66,7.0756,10.6,14.81,2.92,8.5264,11.1,14.81,1.672544e+09,46011_1672544400
2,2023-01-01 04:10:00+00:00,46011,299.0,10.4,13.1,2.99,14.81,7.05,299.0,8.9401,...,2.72,7.3984,8.7,13.79,2.60,6.7600,11.4,11.43,1.672546e+09,46011_1672546200
3,2023-01-01 04:40:00+00:00,46011,301.0,9.8,12.6,3.13,14.81,7.33,285.0,9.7969,...,2.71,7.3441,8.3,11.43,2.33,5.4289,11.1,14.81,1.672548e+09,46011_1672548000
4,2023-01-01 05:10:00+00:00,46011,303.0,9.6,12.7,3.16,13.79,7.25,295.0,9.9856,...,2.79,7.7841,9.4,13.79,2.66,7.0756,10.6,14.81,1.672550e+09,46011_1672549800


In [7]:
# Create Feature Group (unique name per run)
sess = sagemaker.Session()
role = sagemaker.get_execution_role()

FEATURE_GROUP_NAME = f"buoy-wave-features-{int(time.time())}"
OFFLINE_S3_URI = f"s3://{bucket}/feature-store/offline/buoycast/"

ENABLE_ONLINE_STORE = True  # set False if you only want offline

feature_group = FeatureGroup(
    name=FEATURE_GROUP_NAME,
    sagemaker_session=sess,
)

feature_group.load_feature_definitions(data_frame=df_features[
    ["record_id","event_time","station_id"] + feature_cols + ["target_E_star"]
])

print("Creating feature group:", FEATURE_GROUP_NAME)
feature_group.create(
    record_identifier_name="record_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=ENABLE_ONLINE_STORE,
    s3_uri=OFFLINE_S3_URI,
)

import time

def wait_for_feature_group(feature_group, poll_seconds=10, timeout_seconds=600):
    t0 = time.time()
    while True:
        desc = feature_group.describe()
        status = desc.get("FeatureGroupStatus")
        if status in ("Created", "CreateFailed", "DeleteFailed"):
            return desc
        if time.time() - t0 > timeout_seconds:
            raise TimeoutError(f"Timed out waiting for FeatureGroup. Last status: {status}")
        print(f"FeatureGroupStatus: {status} ... waiting")
        time.sleep(poll_seconds)

desc = wait_for_feature_group(feature_group)
print("Final FeatureGroupStatus:", desc["FeatureGroupStatus"])
if desc["FeatureGroupStatus"] != "Created":
    print("FailureReason:", desc.get("FailureReason"))


print("FeatureGroupStatus:", feature_group.describe()["FeatureGroupStatus"])

Creating feature group: buoy-wave-features-1771808190
FeatureGroupStatus: Creating ... waiting
FeatureGroupStatus: Creating ... waiting
Final FeatureGroupStatus: Created
FeatureGroupStatus: Created


In [8]:
# Ingest into Feature Store
ingest_df = df_features[
    ["record_id","event_time","station_id"] + feature_cols + ["target_E_star"]
].copy()

feature_group.ingest(
    data_frame=ingest_df,
    max_workers=4,
    wait=True,
)

desc = feature_group.describe()
print("FeatureGroupStatus:", desc["FeatureGroupStatus"])
print("OfflineStoreStatus:", desc.get("OfflineStoreStatus"))
print("OnlineStoreConfig:", desc.get("OnlineStoreConfig"))

FeatureGroupStatus: Created
OfflineStoreStatus: None
OnlineStoreConfig: {'EnableOnlineStore': True}


In [9]:
# Optional: verify online store retrieval (only works if ENABLE_ONLINE_STORE=True)
if ENABLE_ONLINE_STORE:
    fs_runtime = boto3.client("sagemaker-featurestore-runtime", region_name=region)

    sample_id = ingest_df["record_id"].iloc[0]
    rec = fs_runtime.get_record(FeatureGroupName=FEATURE_GROUP_NAME, RecordIdentifierValueAsString=sample_id)
    rec

In [10]:
# Save for next notebooks
%store FEATURE_GROUP_NAME
%store OFFLINE_S3_URI

Stored 'FEATURE_GROUP_NAME' (str)
Stored 'OFFLINE_S3_URI' (str)
